# CDC with Datastream + a Landing Zone — Hands-On Demo

This notebook builds a **complete, working Change Data Capture pipeline** on a real
Google Cloud project, end to end:

`Cloud SQL for MySQL (small instance)` → `Datastream` → `Cloud Storage landing zone` → `BigQuery (merged current-state table)`

It is the companion to the **"Datastream + Landing Zone Patterns for CDC"** handbook. Each
section below is numbered to match a section in that handbook — if a step doesn't make sense
on its own, the matching handbook section has the full explanation.

## Before you start

- You need a GCP project with **billing enabled** and permission to create Cloud SQL,
  Datastream, Cloud Storage, and BigQuery resources.
- This notebook creates real, billed resources. Run the **Cleanup** cell at the end when
  you're done with the demo.
- The MySQL instance uses `db-f1-micro`, the smallest available Cloud SQL tier, with one small
  demo table — this keeps the footprint (and cost) minimal. It's a training setup, not a
  production sizing (see handbook Section 8.5).
- Expect roughly **15–20 minutes** end to end, mostly spent waiting for the Cloud SQL instance
  to be created and for the Datastream backfill to finish.
- A helper function `run()` is defined in Section 1 and used for every `gcloud` call in this
  notebook — it prints the command, raises on failure, and returns the captured output, so
  each step's outcome is visible rather than hidden inside `!` shell magic.

## 1. Install client libraries and define a small `gcloud` helper

In [ ]:
!pip install --quiet pymysql fastavro pandas google-cloud-storage google-cloud-bigquery

In [ ]:
import subprocess, json, os, time, secrets

def run(cmd, check=True):
    """Run a gcloud/bq/gsutil command (list of args), print it, return stdout."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result.stdout.strip()

## 2. Authenticate and set your project

`auth.authenticate_user()` authenticates this Colab session both for Google Cloud client
libraries **and** for the `gcloud` CLI calls used throughout this notebook.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = ""  #@param {type:"string"}
REGION = "us-central1"  #@param {type:"string"}

assert PROJECT_ID, "Set PROJECT_ID above before continuing."

run(["gcloud", "config", "set", "project", PROJECT_ID])
print(f"Using project: {PROJECT_ID}  region: {REGION}")

## 3. Enable the required APIs\n\nMatches handbook Step 1.

In [ ]:
run(["gcloud", "services", "enable",
     "sqladmin.googleapis.com", "datastream.googleapis.com",
     "storage.googleapis.com", "bigquery.googleapis.com",
     f"--project={PROJECT_ID}"])

In [ ]:
INSTANCE_NAME = "cdc-demo-mysql"
DB_NAME = "demo"
TABLE_NAME = "orders"
BUCKET_NAME = f"{PROJECT_ID}-cdc-landing"
BQ_DATASET = "cdc_demo"
SOURCE_PROFILE = "mysql-source-profile"
DEST_PROFILE = "gcs-landing-profile"
STREAM_NAME = "orders-cdc-stream"

print("Resource names set.")

## 4. Create the smallest usable Cloud SQL for MySQL instance

Matches handbook Step 2. `db-f1-micro` is the smallest shared-core tier — plenty for a demo
table with a handful of CDC events. `--enable-bin-log` is required: Datastream reads the
binary log, and Cloud SQL does not turn it on by default.

This step typically takes **5–10 minutes**.

In [ ]:
ROOT_PASSWORD = secrets.token_urlsafe(16)

run(["gcloud", "sql", "instances", "create", INSTANCE_NAME,
     "--database-version=MYSQL_8_0",
     "--tier=db-f1-micro",
     f"--region={REGION}",
     "--storage-size=10",
     "--storage-type=SSD",
     "--enable-bin-log",
     f"--root-password={ROOT_PASSWORD}",
     f"--project={PROJECT_ID}"])

## 5. Open a network path so this notebook (and Datastream) can reach MySQL

Matches handbook Step 5. For a training lab we use the **public IP + authorized networks**
path (handbook Section 3.3) rather than private connectivity, to avoid VPC setup. Two sets of
IPs need to be authorized:

1. **This Colab runtime's own outbound IP** — so the code below can connect directly with
   `pymysql` to seed data and generate CDC events.
2. **Datastream's static IPs for this region** — fetched live via `gcloud`, so the notebook
   always uses current values instead of a hardcoded list that can go stale (handbook Step 5
   note).

In [ ]:
run(["gcloud", "sql", "instances", "patch", INSTANCE_NAME,
     "--assign-ip", f"--project={PROJECT_ID}", "--quiet"])

In [ ]:
colab_ip = subprocess.run(["curl", "-s", "ifconfig.me"], capture_output=True, text=True).stdout.strip()
print("Colab runtime IP:", colab_ip)

static_ips_out = run(["gcloud", "beta", "datastream", "locations", "fetch-static-ips",
                       f"--location={REGION}", f"--project={PROJECT_ID}", "--format=json"])
datastream_ips = json.loads(static_ips_out).get("staticIps", [])
print("Datastream static IPs for", REGION, ":", datastream_ips)

In [ ]:
authorized_networks = [f"{colab_ip}/32"] + [f"{ip}/32" for ip in datastream_ips]
run(["gcloud", "sql", "instances", "patch", INSTANCE_NAME,
     "--authorized-networks=" + ",".join(authorized_networks),
     f"--project={PROJECT_ID}", "--quiet"])

## 6. Get the instance's public IP, then create the schema, demo table, and Datastream user

Matches handbook Steps 3 and 4. We connect directly with `pymysql` as `root` to:

- create the `demo` database and a small `orders` table (with a primary key — see the
  handbook's note on why that matters for merges),
- seed a couple of starting rows,
- create a `datastream_user` with the minimum replication privileges Datastream needs.

In [ ]:
MYSQL_HOST = run(["gcloud", "sql", "instances", "describe", INSTANCE_NAME,
                   f"--project={PROJECT_ID}", "--format=value(ipAddresses[0].ipAddress)"])
print("Cloud SQL public IP:", MYSQL_HOST)

In [ ]:
import pymysql

DATASTREAM_PASSWORD = secrets.token_urlsafe(16)

conn = pymysql.connect(host=MYSQL_HOST, user="root", password=ROOT_PASSWORD, port=3306, connect_timeout=15)
try:
    with conn.cursor() as cur:
        cur.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME};")
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS {DB_NAME}.{TABLE_NAME} (
              order_id      INT PRIMARY KEY AUTO_INCREMENT,
              customer_name VARCHAR(100),
              amount        DECIMAL(10,2),
              status        VARCHAR(20),
              updated_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
            );
        """)
        cur.execute(f"""
            INSERT INTO {DB_NAME}.{TABLE_NAME} (customer_name, amount, status) VALUES
              ('Asha Rao', 1499.00, 'PLACED'),
              ('Vikram Iyer', 799.50, 'PLACED');
        """)
        cur.execute(
            "CREATE USER IF NOT EXISTS 'datastream_user'@'%%' IDENTIFIED BY %s;",
            (DATASTREAM_PASSWORD,),
        )
        cur.execute("GRANT REPLICATION SLAVE, REPLICATION CLIENT, SELECT ON *.* TO 'datastream_user'@'%';")
        cur.execute("FLUSH PRIVILEGES;")
    conn.commit()
finally:
    conn.close()

print("Schema, table, seed rows, and Datastream user created.")

## 7. Create the landing zone bucket

Matches handbook Step 6 / Section 4.3. This bucket is the raw, append-only **Bronze** layer.

In [ ]:
run(["gcloud", "storage", "buckets", "create", f"gs://{BUCKET_NAME}",
     f"--location={REGION}", "--uniform-bucket-level-access", f"--project={PROJECT_ID}"])

## 8. Create the two Datastream connection profiles

Matches handbook Step 7 — one profile describing how to reach the MySQL source, one
describing the GCS destination.

In [ ]:
run(["gcloud", "datastream", "connection-profiles", "create", SOURCE_PROFILE,
     f"--location={REGION}", "--type=mysql",
     f"--mysql-hostname={MYSQL_HOST}", "--mysql-port=3306",
     "--mysql-username=datastream_user", f"--mysql-password={DATASTREAM_PASSWORD}",
     f"--project={PROJECT_ID}"])

In [ ]:
run(["gcloud", "datastream", "connection-profiles", "create", DEST_PROFILE,
     f"--location={REGION}", "--type=google-cloud-storage",
     f"--bucket={BUCKET_NAME}", "--root-path=/cdc-demo",
     f"--project={PROJECT_ID}"])

## 9. Create and start the stream

Matches handbook Step 8. `--backfill-all` tells Datastream to backfill every included table's
existing rows in addition to streaming ongoing changes (handbook Section 3.1).

In [ ]:
mysql_source_config = json.dumps({
    "includeObjects": {
        "mysqlDatabases": [{"database": DB_NAME, "mysqlTables": [{"table": TABLE_NAME}]}]
    }
})

run(["gcloud", "datastream", "streams", "create", STREAM_NAME,
     f"--location={REGION}",
     f"--source={SOURCE_PROFILE}",
     f"--mysql-source-config={mysql_source_config}",
     f"--destination={DEST_PROFILE}",
     '--gcs-destination-config={"avroFileFormat":{}}',
     "--backfill-all",
     f"--project={PROJECT_ID}"])

In [ ]:
run(["gcloud", "datastream", "streams", "update", STREAM_NAME,
     f"--location={REGION}", "--state=RUNNING", "--update-mask=state",
     f"--project={PROJECT_ID}"])

## 10. Wait for the stream to reach RUNNING

Polls the stream state every 15 seconds. This usually takes a couple of minutes for a table
this small.

In [ ]:
def get_stream_state():
    return run(["gcloud", "datastream", "streams", "describe", STREAM_NAME,
                 f"--location={REGION}", f"--project={PROJECT_ID}", "--format=value(state)"],
                check=False)

for _ in range(40):
    state = get_stream_state()
    print("Stream state:", state)
    if state == "RUNNING":
        break
    time.sleep(15)
else:
    print("Stream did not reach RUNNING in the expected time — check the Datastream console for errors.")

## 11. Generate live changes

Matches handbook Step 9. With the stream running, every insert/update/delete below should
show up as a new Avro file in the landing bucket within roughly seconds to a couple of
minutes.

In [ ]:
conn = pymysql.connect(host=MYSQL_HOST, user="root", password=ROOT_PASSWORD, port=3306, connect_timeout=15)
try:
    with conn.cursor() as cur:
        cur.execute(f"INSERT INTO {DB_NAME}.{TABLE_NAME} (customer_name, amount, status) VALUES ('Meera Nair', 2199.00, 'PLACED');")
        cur.execute(f"UPDATE {DB_NAME}.{TABLE_NAME} SET status='SHIPPED' WHERE customer_name='Asha Rao';")
        cur.execute(f"DELETE FROM {DB_NAME}.{TABLE_NAME} WHERE customer_name='Vikram Iyer';")
    conn.commit()
finally:
    conn.close()

print("Generated an insert, an update, and a delete. Waiting a minute for Datastream to capture them...")
time.sleep(60)

## 12. Inspect the landing zone

Matches handbook Step 10 (first half). Lists the Avro files Datastream wrote, downloads them,
and reads them into a DataFrame — including the `_metadata_change_type` column that tells you
whether each row is an insert, update, or delete event (handbook Section 3.4).

In [ ]:
from google.cloud import storage
import fastavro
import pandas as pd
import io

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)
blobs = list(bucket.list_blobs(prefix="cdc-demo/"))
avro_blobs = [b for b in blobs if b.name.endswith(".avro")]
print(f"Found {len(avro_blobs)} Avro file(s) in the landing zone.")

records = []
for blob in avro_blobs:
    data = blob.download_as_bytes()
    for record in fastavro.reader(io.BytesIO(data)):
        records.append(record)

landing_df = pd.DataFrame(records)
landing_df

If `avro_blobs` is empty, the backfill/CDC files may not have landed yet — wait another minute
and re-run this cell. If it's still empty, check the stream's status in the Datastream console
for connectivity errors (handbook Section 10, Troubleshooting Guide).

## 13. Merge the landing zone into a current-state BigQuery table

Matches handbook Step 10 (second half) / Section 4.2's Silver layer. We load the raw change
events into a staging table, then run a `MERGE` keyed on `order_id`, keeping only the latest
version of each row and dropping rows whose latest event is a delete — demonstrating the
mechanics by hand, the same operation Datastream's built-in BigQuery destination runs for you
automatically.

In [ ]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{BQ_DATASET}")
dataset_ref.location = REGION
bq.create_dataset(dataset_ref, exists_ok=True)

staging_table_id = f"{PROJECT_ID}.{BQ_DATASET}.orders_staging"
current_table_id = f"{PROJECT_ID}.{BQ_DATASET}.orders_current"

stage_cols = ["order_id", "customer_name", "amount", "status", "updated_at",
              "_metadata_change_type", "_metadata_timestamp"]
stage_df = landing_df[[c for c in stage_cols if c in landing_df.columns]].copy()

job = bq.load_table_from_dataframe(
    stage_df, staging_table_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()
print(f"Loaded {len(stage_df)} raw change events into {staging_table_id}")

In [ ]:
merge_sql = f"""
CREATE TABLE IF NOT EXISTS `{current_table_id}` (
  order_id INT64, customer_name STRING, amount NUMERIC, status STRING, updated_at TIMESTAMP
);

MERGE `{current_table_id}` T
USING (
  SELECT * EXCEPT(rn) FROM (
    SELECT *,
      ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY _metadata_timestamp DESC) AS rn
    FROM `{staging_table_id}`
  )
  WHERE rn = 1
) S
ON T.order_id = S.order_id
WHEN MATCHED AND S._metadata_change_type = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET
  customer_name = S.customer_name, amount = S.amount, status = S.status, updated_at = S.updated_at
WHEN NOT MATCHED AND S._metadata_change_type != 'DELETE' THEN
  INSERT (order_id, customer_name, amount, status, updated_at)
  VALUES (S.order_id, S.customer_name, S.amount, S.status, S.updated_at);
"""
bq.query(merge_sql).result()
print("Merge complete.")

result_df = bq.query(f"SELECT * FROM `{current_table_id}` ORDER BY order_id").to_dataframe()
result_df

You should now see **Meera Nair** (insert), **Asha Rao** with status `SHIPPED` (update), and
**no row for Vikram Iyer** (delete) — the current-state table reflects every change captured
through the log, without ever running a query against the live `orders` table itself.

Try inserting/updating/deleting more rows in Section 11 and re-running Sections 12–13 to watch
the landing zone and the merged table pick up new changes.

## 14. Cleanup

Run this when you're done to avoid ongoing charges. It deletes, in dependency order: the
stream, both connection profiles, the Cloud SQL instance, the landing bucket, and the BigQuery
dataset. **This is destructive and cannot be undone** — the checkbox below is a safety guard.

In [ ]:
CONFIRM_CLEANUP = False  #@param {type:"boolean"}

if CONFIRM_CLEANUP:
    run(["gcloud", "datastream", "streams", "update", STREAM_NAME,
         f"--location={REGION}", "--state=PAUSED", "--update-mask=state",
         f"--project={PROJECT_ID}", "--quiet"], check=False)
    run(["gcloud", "datastream", "streams", "delete", STREAM_NAME,
         f"--location={REGION}", f"--project={PROJECT_ID}", "--quiet"], check=False)
    run(["gcloud", "datastream", "connection-profiles", "delete", SOURCE_PROFILE,
         f"--location={REGION}", f"--project={PROJECT_ID}", "--quiet"], check=False)
    run(["gcloud", "datastream", "connection-profiles", "delete", DEST_PROFILE,
         f"--location={REGION}", f"--project={PROJECT_ID}", "--quiet"], check=False)
    run(["gcloud", "sql", "instances", "delete", INSTANCE_NAME,
         f"--project={PROJECT_ID}", "--quiet"], check=False)
    run(["gcloud", "storage", "rm", "-r", f"gs://{BUCKET_NAME}",
         f"--project={PROJECT_ID}"], check=False)
    run(["bq", "rm", "-r", "-f", "-d", f"{PROJECT_ID}:{BQ_DATASET}"], check=False)
    print("Cleanup complete.")
else:
    print("Set CONFIRM_CLEANUP = True above and re-run this cell to tear down all resources.")